# LBM-Suite2p Quickstart

``` {note}
Example dataset collected by kevin barber with Dr. Alipasha Vaziri @rockefeller university.

Animal: mk301
Date:  2025-03-01
Virus: jGCaMP8s
Framerate: 17hz
FOV: 900um x 900um
Resolution: 2um x 2um x 16um
```

In [2]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import numpy as np
import suite2p
import mbo_utilities as mbo
import fastplotlib as fpl
from copy import deepcopy
import lbm_suite2p_python as lsp

See the [assembly documentation](https://millerbrainobservatory.github.io/mbo_utilities/assembly.html) for a guide on extracting data before input into suite2p.

``` python
scan = mbo.read_scan(r"path/to/tiffs*") # glob or list of filepaths (see mbo.get_files)
mbo.save_as(scan, "/path/to/save") # defaults to saving plane_01.tiff, plane_02.tiff ... 
```
Suite2p is primarily a 2D pipeline - we will run each z-plane sequentially and combine results at the end.

In [5]:
# most pipelines have a "parent" folder to save files relative to
animal_path = Path(r"D:\W2_DATA\kbarber\2025_03_01\mk301")  # (optional) the parent directory for this session
assembled_path = animal_path.joinpath("assembled")          # where our assembled tiffs live
animal_path.is_dir(), assembled_path.is_dir()

(True, True)

## Input tifs

The tifs we use as input are planar timeseries `[Txy]`. Raw ScanImage tiffs **will not work here**, as they are not in the correct frame order. 

In [13]:
input_files = mbo.get_files(assembled_path, str_contains='tif', max_depth=3)
[Path(x).parts[-2:] for x in input_files[:3]] # show just the first 3 files

[('assembled', 'plane_01_mk301.tiff'),
 ('assembled', 'plane_02_mk301.tiff'),
 ('assembled', 'plane_03_mk301.tiff')]

## Default metadata

{func}`mbo_utilities.get_metadata()` will retrieve the ScanImage metadata (frame rate, pixel resolution, image dimensions).

We then feed this metadata into {func}`mbo_utilities.params_from_metadata` to autofill the suite2p parameters that rely on these metadata.

In [7]:
metadata = mbo.get_metadata(input_files[0])
ops = suite2p.default_ops()
ops = mbo.params_from_metadata(metadata, ops)

# we filled in pixel resolution and frame rate
ops["dx"], ops["dy"], ops["fs"]

Ops provided. Setting pipeline to suite2p


([2.0], [2.0], 17.06701142272251)

## Process single z-plane

In [25]:
save_path = Path("./results")
save_path.mkdir(exist_ok=True)
print(f"Saving suite2p results to:  {save_path.resolve()}")

Saving suite2p results to:  D:\W2_DATA\kbarber\2025_03_01\mk301\scripts\results


In [36]:
input_file = Path(input_files[7]) # pick a zplane in the middle of the cavity for example
input_file

WindowsPath('D:/W2_DATA/kbarber/2025_03_01/mk301/assembled/plane_08_mk301.tiff')

### Default opts 

For this demo, we use the the default ops provided by {func}`suite2p.default_ops`.

In [ ]:
ops = lsp.run_plane(
    ops=ops,
    input_tiff=input_file,
    save_path=save_path,
    save_folder = str(input_file.stem),  # strip the path and extension from this filename
    replot=True
)

## Planar Outputs

Standard suite2p files:

- ops.npy
- stat.npy
- spks.npy
- iscell.npy
- F.npy
- Fneu.npy

Plots are created by default::

- summary images (max-projection, mean-image)
- Accepted/rejected masks drawn on summary image
- 20 randomly selected DF/F traces

In [41]:
list(save_path.rglob("*"))

[WindowsPath('results/plane_08_mk301'),
 WindowsPath('results/plane_08_mk301/plane0'),
 WindowsPath('results/plane_08_mk301/plane0/F.npy'),
 WindowsPath('results/plane_08_mk301/plane0/Fneu.npy'),
 WindowsPath('results/plane_08_mk301/plane0/iscell.npy'),
 WindowsPath('results/plane_08_mk301/plane0/max_projection_image.png'),
 WindowsPath('results/plane_08_mk301/plane0/mean_image.png'),
 WindowsPath('results/plane_08_mk301/plane0/ops.npy'),
 WindowsPath('results/plane_08_mk301/plane0/segmentation.png'),
 WindowsPath('results/plane_08_mk301/plane0/spks.npy'),
 WindowsPath('results/plane_08_mk301/plane0/stat.npy'),
 WindowsPath('results/plane_08_mk301/plane0/traces.png')]

### Run full volume

To run the entire volume, {func}`lbm_suite2p_python.run_volume` takes the same inputs as its planar varient, except give a list of input files rather than a single input tiff file.

In [ ]:
output_ops = lsp.run_volume(ops, input_files, save_path)

In [ ]:
ops_files = mbo.get_files(save_path.parent, 'ops', 8)

In [24]:
stat_files = mbo.get_files(save_path.parent, 'stat.npy', max_depth=5)
stat_files[:3]

['D:\\W2_DATA\\kbarber\\2025-02-10\\mk303\\results\\plane_01_demo\\plane0\\stat.npy',
 'D:\\W2_DATA\\kbarber\\2025-02-10\\mk303\\results\\plane_02_demo\\plane0\\stat.npy',
 'D:\\W2_DATA\\kbarber\\2025-02-10\\mk303\\results\\plane_03_demo\\plane0\\stat.npy']

In [ ]:
output_ops = lsp.run_volume(ops, input_files, save_path=save_path, replot=True)